# LLM Pipeline to Process Extracted Dataset (Phase 1)

In [3]:
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


**Locate extracted dataset**

In [2]:
from pathlib import Path

def find_dataset(dirname: str, search_root=None):
    root = search_root or Path.cwd()

    direct = root / dirname

    return direct if direct.is_dir() else None

In [19]:
DATASET_NAME = "extracted_dataset_1792212375"
DATASET_PATH = find_dataset(DATASET_NAME)

#  Define DISEASES by scanning the directories in DATASET_PATH
DISEASES = sorted([item.name for item in DATASET_PATH.iterdir() if item.is_dir()])

In [20]:
print(DISEASES)

['brucellosis', 'covid-19', 'dengue', 'extrapulmonary tuberculosis', 'hepatitis', 'hepatitis b', 'hepatitis c', 'leprosy', 'leptospirosis', 'malaria', 'melioidosis', 'mycobacterium tuberculosis', 'neurocysticercosis', 'pulmonary tuberculosis', 'q fever', 'tuberculosis']


**Helper functions to rename and retrieve cases**

In [21]:
import pandas as pd

def rename_cases(disease_name):
    disease_folder = DATASET_PATH / disease_name
    old_file = disease_folder / "cases.csv"
    new_file = disease_folder / f"{disease_name}_cases.csv"

    if old_file.exists():
        old_file.rename(new_file)
    else:
        print(f"File {old_file} does not exist or has already been renamed.")

def retrieve_cases(disease_name):
    disease_folder = DATASET_PATH / disease_name
    csv_path = disease_folder / f"{disease_name}_cases.csv"

    # Load cases.csv and extract specific columns into a pandas DataFrame
    df = pd.read_csv(csv_path)

    return df

In [ ]:
# Rename once
for disease in DISEASES:
    rename_cases(disease_name=disease)

**Helper functions to delete specific cases (via `case_id`)**

In [23]:
def delete_img_files(disease_name, case_id):
    """
    Reads the image_metadata.json for a specific disease, finds entries matching
    the case_id, modifies their file paths, and deletes the corresponding image files.
    The entry in the JSON file is removed if and only if the image file is successfully deleted.
    """
    metadata_path = DATASET_PATH / disease_name / "image_metadata.json"

    if not metadata_path.exists():
        print(f"Metadata file not found at {metadata_path}")
        return

    lines_to_keep = []

    # Read and process the JSON lines
    with open(metadata_path, 'r') as file:
        for line in file:
            line_str = line.strip()
            if not line_str:
                continue

            record = json.loads(line_str)

            # Check if the row matches the target case_id
            if record.get("case_id") == case_id:
                original_path = record.get("file_path")

                if original_path and original_path.startswith("medical_datasets/"):
                    relative_path = original_path.split('/', 1)[1]
                    img_path_to_delete = DATASET_PATH / relative_path

                    try:
                        # Attempt to delete the file
                        if img_path_to_delete.exists() and img_path_to_delete.is_file():
                            img_path_to_delete.unlink()
                            print(f"Successfully deleted: {img_path_to_delete}")
                        else:
                            print(f"Image not found at path: {img_path_to_delete}. Metadata entry retained.")

                        # Skip appending this line to 'lines_to_keep' since deletion succeeded
                        continue
                    except Exception as e:
                        print(f"Error deleting image {img_path_to_delete}: {e}. Metadata entry retained.")

            # If case_id doesn't match, or the image deletion failed/was not found, keep the line
            lines_to_keep.append(line)

    # Write the remaining lines back to the file
    with open(metadata_path, 'w') as file:
        file.writelines(lines_to_keep)


def delete_cases(disease_name, case_id):
    """
    Deletes all entries in the disease-specific cases.csv where case_id matches.
    """
    csv_path = DATASET_PATH / disease_name / f"{disease_name}_cases.csv"

    if not csv_path.exists():
        print(f"Cases CSV file not found at {csv_path}")
        return

    # Load the CSV file into a DataFrame[cite: 1]
    df = pd.read_csv(csv_path)

    if 'case_id' not in df.columns:
        print(f"'case_id' column missing in {csv_path}")
        return

    initial_row_count = len(df)

    # Filter out the rows matching the case_id
    df_filtered = df[df['case_id'] != case_id]

    final_row_count = len(df_filtered)

    # Save the updated DataFrame back to CSV if rows were removed
    if initial_row_count > final_row_count:
        df_filtered.to_csv(csv_path, index=False)
        print(f"Successfully deleted {initial_row_count - final_row_count} row(s) for case_id: {case_id}")
    else:
        print(f"No matching entries found for case_id: {case_id}")

**Eliminate cases with unclear demographic**

In [25]:
def flag_unknown_demographics(df):
    """
    Flags cases where age is NaN or gender is NaN/'Unknown'.
    Returns a list of dicts containing 'case_id' and 'pmc_id'.
    """
    # Create a mask for missing age or missing/unknown gender
    mask = (
        df['age'].isna() |
        df['gender'].isna() |
        (df['gender'].astype(str).str.lower() == 'unknown')
    )

    # Filter the dataframe and select relevant columns[cite: 1]
    flagged_df = df[mask][['case_id', 'pmcid']]

    flagged_df = flagged_df.rename(columns={'pmcid': 'pmc_id'})

    # Convert to a list of dictionaries
    return flagged_df.to_dict(orient='records')

In [26]:
for item in DATASET_PATH.iterdir():
    if item.is_dir():
        # 1) Record the name of each folder in a variable, called disease
        disease = item.name

        try:
            # 2) Load the cases.csv file inside that folder using retrieve_cases()
            df = retrieve_cases(disease)

            # 3) Fetch all case_ids without clear demographics
            flagged_cases = flag_unknown_demographics(df)

            # 4) Delete images, metadata, and case records for the flagged cases
            for case in flagged_cases:
                target_case_id = case['case_id']

                # Delete images and the metadata entry in image_metadata.json
                delete_img_files(disease, target_case_id)

                # Delete the specific case entry in the cases.csv file
                delete_cases(disease, target_case_id)

        except Exception as e:
            print(f"An error occurred while processing '{disease}': {e}")

**Normalize disease names using disease ontology (UMLS)**

In [27]:
from dotenv import dotenv_values

env_path = env_path = Path.cwd() / "public.env"
UMLS_API_KEY = dotenv_values(env_path).get("UMLS_API_KEY")

In [28]:
import requests
import json


def search_umls(query_string, api_key=UMLS_API_KEY, ):
    """Searches UMLS for a specific string and returns basic results."""

    # The base URL for the UMLS REST API search endpoint
    url = "https://uts-ws.nlm.nih.gov/rest/search/current"

    # The parameters for the GET request
    params = {
        "string": query_string,
        "apiKey": api_key,
        "pageNumber": 1 # Optional: defaults to 1
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # Raise an exception for bad status codes

        data = response.json()

        # Extract the results from the JSON response
        results = data.get("result", {}).get("results", [])

        if not results:
            print(f"No results found for '{query_string}'")
            return

        return results[0]['name']

    except requests.exceptions.RequestException as e:
        print(f"Error communicating with the UMLS API: {e}")

In [31]:
# Normalize disease names and save to this list
NORMALIZED_DISEASES = []

for disease in DISEASES:
    normalized_name = search_umls(disease)

    NORMALIZED_DISEASES.append({
        "original": disease,
        "normalized": normalized_name
    })

In [32]:
# Save to dedicated .csv file
df_normalized = pd.DataFrame(NORMALIZED_DISEASES)

# Pandas automatically handles fields with commas by enclosing them in quotes
df_normalized.to_csv(DATASET_PATH / "normalized_diseases.csv", index=False)